In [ ]:
from graphein.ml.conversion import GraphFormatConvertor
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from graphein.protein.utils import get_obsolete_mapping
import pandas as pd
import os
from tqdm.notebook import tqdm

In [ ]:
from tqdm.notebook import tqdm
import time

for i in tqdm(range(10)):
    time.sleep(0.1)

In [ ]:
# CONFIGS
import graphein.protein as gp
from functools import partial
from graphein.ml.conversion import GraphFormatConvertor
from graphein.protein.edges.distance import (add_peptide_bonds,
                                             add_hydrogen_bond_interactions,
                                             add_disulfide_interactions,
                                             add_ionic_interactions,
                                             add_aromatic_interactions,
                                             add_aromatic_sulphur_interactions,
                                             add_cation_pi_interactions
                                             )


# 1: Distance-based
dist_edge_func = {"edge_construction_functions": [partial(gp.add_distance_threshold, threshold=5, long_interaction_threshold=0)]}

# 2: Biochemical interactions, select set
select_edge_func = {"edge_construction_functions": [add_peptide_bonds,
                                                    add_hydrogen_bond_interactions,
                                                    add_disulfide_interactions,
                                                    add_ionic_interactions,
                                                    gp.add_salt_bridges]}

# 3: Biochemical interactions, expanded set
all_edge_func = {"edge_construction_functions": [add_peptide_bonds,
                                                 add_aromatic_interactions,
                                                 add_hydrogen_bond_interactions,
                                                 add_disulfide_interactions,
                                                 add_ionic_interactions,
                                                 add_aromatic_sulphur_interactions,
                                                 add_cation_pi_interactions,
                                                 gp.add_hydrophobic_interactions,
                                                 gp.add_vdw_interactions,
                                                 gp.add_backbone_carbonyl_carbonyl_interactions,
                                                 gp.add_salt_bridges]}

In [ ]:

# A: Just one-hot encodings
one_hot = {"node_metadata_functions" : [gp.amino_acid_one_hot]}

# B: Selected biochemical features
all_graph_metadata = {"graph_metadata_functions" : [gp.rsa,
                                                    gp.secondary_structure]}
all_node_metadata = {"node_metadata_functions" : [gp.amino_acid_one_hot,
                                                  gp.meiler_embedding,
                                                  partial(gp.expasy_protein_scale, add_separate=True)],
                     #"dssp_config": gp.DSSPConfig()
                     }


config_1A = gp.ProteinGraphConfig(**{**dist_edge_func, **one_hot})
config_1B = gp.ProteinGraphConfig(**{**dist_edge_func, **all_graph_metadata, **all_node_metadata})

config_2A = gp.ProteinGraphConfig(**{**select_edge_func, **one_hot})
config_2B = gp.ProteinGraphConfig(**{**select_edge_func, **all_graph_metadata, **all_node_metadata})

config_3A = gp.ProteinGraphConfig(**{**all_edge_func, **one_hot})
config_3B = gp.ProteinGraphConfig(**{**all_edge_func, **all_graph_metadata, **all_node_metadata})

In [ ]:
# Plotting
from graphein.protein.graphs import construct_graph
from graphein.protein.visualisation import plotly_protein_structure_graph

g1 = construct_graph(config=config_1A, path="6RAI.pdb")
g2 = construct_graph(config=config_2A, path="6RAK.pdb")
g3 = construct_graph(config=config_3A, path="6RVC.pdb")

p1 = plotly_protein_structure_graph(
    g1,
    colour_edges_by="kind",
    colour_nodes_by="degree",
    label_node_ids=False,
    plot_title="",
    node_size_multiplier=1
)
p2 = plotly_protein_structure_graph(
    g2,
    colour_edges_by="kind",
    colour_nodes_by="degree",
    label_node_ids=False,
    plot_title="",
    node_size_multiplier=1
)
p3 = plotly_protein_structure_graph(
    g3,
    colour_edges_by="kind",
    colour_nodes_by="degree",
    label_node_ids=False,
    plot_title="",
    node_size_multiplier=1
)

In [ ]:
p1.show(); p2.show(); p3.show()

In [ ]:
# The graphs are stored in memory, so if you want to generate new graphs with a
# different config, you need to delete the old ones first.
#%rm -r processed
%rm -r data

In [ ]:
from graphein.ml import ProteinGraphListDataset, GraphFormatConvertor
import graphein.protein as gp

# Construct graphs
graphs = gp.construct_graphs_mp(
    pdb_code_it=["6RAI", "6RAK", "6RVC"],
    return_dict=False
)

# CHOOSE CONFIG FILE:
config = config_3B #1A is least memory-intensive
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index",
                                                                             "amino_acid_one_hot", "bulkiness",
                                                                             "meiler", "rsa", "pka_rgroup",
                                                                             "isoelectric_points", "polaritygrantham",
                                                                             "hphob_black", "transmembranetendency"])

graphs = [convertor(g) for g in graphs]

# Create dataset
ds = ProteinGraphListDataset(root=".", data_list=graphs, name="list_test")


In [ ]:
# First, check what's available in your Graphein installation
import graphein.protein as gp

# Check if these collections are defined
print("Checking configuration variables:")
if 'all_edge_func' in globals():
    print("all_edge_func is defined")
else:
    print("all_edge_func is NOT defined")

if 'all_graph_metadata' in globals():
    print("all_graph_metadata is defined")
else:
    print("all_graph_metadata is NOT defined")

if 'all_node_metadata' in globals():
    print("all_node_metadata is defined")
else:
    print("all_node_metadata is NOT defined")

# If these aren't defined, you need to define them first
# Example (adjust based on your Graphein version):
# from graphein.protein.config import all_edge_funcs as all_edge_func
# from graphein.protein.config import all_graph_metadata_funcs as all_graph_metadata
# from graphein.protein.config import all_node_metadata_funcs as all_node_metadata

# Now, construct the graphs
config = gp.ProteinGraphConfig(**{**all_edge_func, **all_graph_metadata, **all_node_metadata})
graphs = gp.construct_graphs_mp(
    pdb_code_it=["6RAI", "6RAK", "6RVC"],
    config=config,
    return_dict=False
)

# Inspect the first graph to see available features
nx_graph = graphs[0]
sample_node = list(nx_graph.nodes())[0]
print(f"\nAvailable node features in NetworkX graph:")
for key in nx_graph.nodes[sample_node].keys():
    print(f"- {key}: {type(nx_graph.nodes[sample_node][key])}")

# Now convert with all available features
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index",
                                                                             "amino_acid_one_hot", "bulkiness",
                                                                             "meiler", "rsa", "pka_rgroup",
                                                                             "isoelectric_points", "polaritygrantham",
                                                                             "hphob_black", "transmembranetendency"])
pyg_graphs = [convertor(g) for g in graphs]

# Check what ended up in the PyG graph
pyg_graph = pyg_graphs[0]
print(f"\nAvailable features in PyG graph:")
for key in pyg_graph.keys:
    if hasattr(pyg_graph[key], 'shape'):
        print(f"- {key}: {pyg_graph[key].shape}")
    else:
        print(f"- {key}: {type(pyg_graph[key])}")

# Create dataset
ds = ProteinGraphListDataset(root=".", data_list=pyg_graphs, name="list_test")

for b in ds:
    print(b)
    break

In [ ]:
for b in ds:
    print("Available keys:", b.keys)

    # Print shapes of specific features you want to see
    if hasattr(b, 'amino_acid_one_hot'):
        print("amino_acid_one_hot shape:", b.amino_acid_one_hot.shape)

    if hasattr(b, 'bulkiness'):
        print("bulkiness shape:", b.bulkiness.shape)

    if hasattr(b, 'rsa'):
        print("rsa shape:", b.rsa.shape)

    # Check first few values of some features
    if hasattr(b, 'amino_acid_one_hot'):
        print("First amino one-hot:", b.amino_acid_one_hot[0])

    if hasattr(b, 'bulkiness'):
        print("First few bulkiness values:", b.bulkiness[:5])

    break

In [ ]:
# Your original code
from graphein.ml import ProteinGraphListDataset, GraphFormatConvertor
import graphein.protein as gp

# Construct graphs with your comprehensive config
graphs = gp.construct_graphs_mp(
    pdb_code_it=["6RAI", "6RAK", "6RVC"],
    config=config_3B,  # This config should include all feature functions
    return_dict=False
)

# Create a convertor but don't specify columns - use all available ones
convertor = GraphFormatConvertor(
    src_format="nx",
    dst_format="pyg"
    # Not specifying columns will use all available attributes
)

# Convert your graphs
pyg_graphs = [convertor(g) for g in graphs]

# Create dataset
ds = ProteinGraphListDataset(root=".", data_list=pyg_graphs, name="list_test")

# Check features in the created dataset
sample_graph = ds[0]
print("Features in the PyG dataset:")
for key in sample_graph.keys:
    if hasattr(sample_graph[key], 'shape'):
        print(f"- {key}: {sample_graph[key].shape}")
    else:
        print(f"- {key}: {type(sample_graph[key])}")

In [ ]:
# First print out general dataset info
print(f"Dataset length: {len(ds)}")
print(f"Dataset name: {ds.name}")

# Examine properties of the first graph in the dataset
sample_graph = ds[0]
print("\nProperties of first graph:")
for key in sample_graph.keys:
    print(f"- {key}: {sample_graph[key].shape if hasattr(sample_graph[key], 'shape') else sample_graph[key]}")

# Print out node features
print("\nNode features:")
for key in sample_graph.keys:
    if isinstance(sample_graph[key], torch.Tensor) and len(sample_graph[key].shape) == 2 and sample_graph[key].shape[0] == sample_graph.num_nodes:
        print(f"- {key}: {sample_graph[key].shape}")

# Print out edge features
print("\nEdge features:")
for key in sample_graph.keys:
    if isinstance(sample_graph[key], torch.Tensor) and len(sample_graph[key].shape) == 2 and sample_graph[key].shape[0] == 2:
        print(f"- {key}: {sample_graph[key].shape}")

# Print data types
print("\nData types:")
for key in sample_graph.keys:
    print(f"- {key}: {type(sample_graph[key])}")

In [ ]:
# First, check if the NetworkX graph nodes have the features you want
nx_graph = graphs[0]
sample_node = list(nx_graph.nodes())[0]

# Get all available node attributes from the first node
node_attrs = list(nx_graph.nodes[sample_node].keys())

# Create a custom convertor that doesn't use the standard column approach
# Instead, manually create a PyG Data object with the features you want
import torch
from torch_geometric.data import Data

def custom_nx_to_pyg(nx_graph):
    # Get node features and convert to tensors
    num_nodes = len(nx_graph.nodes)
    node_ids = []
    coords_list = []
    meiler_list = []
    amino_acid_one_hot_list = []
    bulkiness_list = []
    rsa_list = []
    pka_rgroup_list = []
    isoelectric_points_list = []
    polaritygrantham_list = []
    hphob_black_list = []
    transmembranetendency_list = []

    for node in nx_graph.nodes():
        node_data = nx_graph.nodes[node]
        node_ids.append(node_data.get('chain_id', '') + ':' +
                        node_data.get('residue_name', '') + ':' +
                        str(node_data.get('residue_number', '')))

        # Add coordinates
        coords_list.append(node_data.get('coords', [0, 0, 0]))

        # Add Meiler features
        meiler = node_data.get('meiler')
        if meiler is not None:
            if hasattr(meiler, 'values'):  # If it's a pandas Series
                meiler_list.append(meiler.values)
            else:
                meiler_list.append(meiler)
        else:
            meiler_list.append([0] * 7)  # Default meiler has 7 features

        # Add other features - handle both scalar values and arrays
        amino_one_hot = node_data.get('amino_acid_one_hot')
        if amino_one_hot is not None:
            amino_acid_one_hot_list.append(amino_one_hot)
        else:
            amino_acid_one_hot_list.append([0] * 20)  # Default one-hot size

        # Add scalar features with default 0
        bulkiness_list.append(node_data.get('bulkiness', 0))
        rsa_list.append(node_data.get('rsa', 0))
        pka_rgroup_list.append(node_data.get('pka_rgroup', 0))
        isoelectric_points_list.append(node_data.get('isoelectric_points', 0))
        polaritygrantham_list.append(node_data.get('polaritygrantham', 0))
        hphob_black_list.append(node_data.get('hphob_black', 0))
        transmembranetendency_list.append(node_data.get('transmembranetendency', 0))

    # Create tensors for PyG
    coords = torch.tensor(coords_list, dtype=torch.float)
    meiler = torch.tensor(meiler_list, dtype=torch.float)

    # Try to convert amino acid one hot to tensor (might need different handling)
    try:
        amino_acid_one_hot = torch.tensor(amino_acid_one_hot_list, dtype=torch.float)
    except:
        amino_acid_one_hot = None

    # Convert scalar features to tensors
    bulkiness = torch.tensor(bulkiness_list, dtype=torch.float).view(-1, 1)
    rsa = torch.tensor(rsa_list, dtype=torch.float).view(-1, 1)
    pka_rgroup = torch.tensor(pka_rgroup_list, dtype=torch.float).view(-1, 1)
    isoelectric_points = torch.tensor(isoelectric_points_list, dtype=torch.float).view(-1, 1)
    polaritygrantham = torch.tensor(polaritygrantham_list, dtype=torch.float).view(-1, 1)
    hphob_black = torch.tensor(hphob_black_list, dtype=torch.float).view(-1, 1)
    transmembranetendency = torch.tensor(transmembranetendency_list, dtype=torch.float).view(-1, 1)

    # Get edge indices
    edge_index = []
    for u, v in nx_graph.edges():
        edge_index.append([nx_graph.nodes[u].get('residue_number', 0),
                           nx_graph.nodes[v].get('residue_number', 0)])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Create PyG Data object
    data = Data(
        coords=coords,
        meiler=meiler,
        node_id=node_ids,
        edge_index=edge_index,
        num_nodes=num_nodes
    )

    # Add other features if they were successfully converted
    if amino_acid_one_hot is not None:
        data.amino_acid_one_hot = amino_acid_one_hot

    data.bulkiness = bulkiness
    data.rsa = rsa
    data.pka_rgroup = pka_rgroup
    data.isoelectric_points = isoelectric_points
    data.polaritygrantham = polaritygrantham
    data.hphob_black = hphob_black
    data.transmembranetendency = transmembranetendency

    return data

# Convert all graphs using custom converter
pyg_graphs = [custom_nx_to_pyg(g) for g in graphs]

# Create dataset
ds = ProteinGraphListDataset(root=".", data_list=pyg_graphs, name="list_test")

# Check what features are in the converted graphs
sample_graph = ds[0]
print("Features in custom PyG dataset:")
for key in sample_graph.keys:
    if hasattr(sample_graph[key], 'shape'):
        print(f"- {key}: {sample_graph[key].shape}")
    else:
        print(f"- {key}: {type(sample_graph[key])}")

In [ ]:
from torch_geometric.data import DataLoader

train_loader = DataLoader(ds, batch_size=3, shuffle=True, drop_last=True)


In [ ]:
for b in ds:
    print(b)
    break

In [ ]:
%rm -r data


In [ ]:
%rm -r data


from graphein.ml import InMemoryProteinGraphDataset
import os
# 1: Distance-based
dist_edge_func = {"edge_construction_functions": [partial(gp.add_distance_threshold, threshold=5, long_interaction_threshold=0)]}

# 2: Biochemical interactions, select set
select_edge_func = {"edge_construction_functions": [add_peptide_bonds,
                                                    add_hydrogen_bond_interactions,
                                                    ]}

# 3: Biochemical interactions, expanded set
all_edge_func = {"edge_construction_functions": [add_peptide_bonds,
                                                 add_aromatic_interactions,
                                                 add_hydrogen_bond_interactions,
                                                 add_disulfide_interactions,
                                                 add_ionic_interactions,
                                                 add_aromatic_sulphur_interactions,
                                                 add_cation_pi_interactions,
                                                 gp.add_hydrophobic_interactions,
                                                 gp.add_vdw_interactions,
                                                 gp.add_backbone_carbonyl_carbonyl_interactions,
                                                 gp.add_salt_bridges]}

# A: Just one-hot encodings
one_hot = {"node_metadata_functions" : [gp.amino_acid_one_hot, gp.meiler_embedding,
                                        partial(gp.expasy_protein_scale, add_separate=True)]}

# B: Selected biochemical features
all_graph_metadata = {"graph_metadata_functions" : [gp.rsa,
                                                    gp.secondary_structure]}
all_node_metadata = {"node_metadata_functions" : [gp.amino_acid_one_hot,
                                                  gp.meiler_embedding,
                                                  partial(gp.expasy_protein_scale, add_separate=True)],
                     #"dssp_config": gp.DSSPConfig()
                     }


config_1A = gp.ProteinGraphConfig(**{**dist_edge_func, **one_hot})
config = config_1A #1A is least memory-intensive
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index",
                                                                             "amino_acid_one_hot", "bulkiness",
                                                                             "meiler", "rsa", "pka_rgroup",
                                                                             "isoelectric_points", "polaritygrantham",
                                                                             "hphob_black", "transmembranetendency"])

# Get paths to all your PDB files
pdb_dir = os.path.expanduser("~/Downloads/nanobody_extracted2")
pdb_paths = [os.path.join(pdb_dir, f) for f in os.listdir(pdb_dir) if f.endswith('.pdb')]

# Create label map (assuming you have a way to determine labels)
# For example, if nanobody in filename means label=1:
label_map = {os.path.splitext(os.path.basename(path))[0]: 1 if "nanobody" in path else 0
             for path in pdb_paths}

# Create the dataset
train_ds = InMemoryProteinGraphDataset(
    root="data/",
    name="train",
    paths=pdb_paths,  # Use paths instead of pdb_codes
    graph_label_map=label_map,
    graphein_config=config_1A,  # Use whichever config you prefer
    graph_format_convertor=convertor,
    graph_transformation_funcs=[],
)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, drop_last=True)


In [ ]:
for b in train_ds:
    print(b)
    break

In [ ]:
num_proteins = len(train_ds)
print(f"Number of proteins in the dataset: {num_proteins}")

In [ ]:
import pandas as pd

def protein_to_dataframe(protein_data):
    """Convert a PyG protein data object to a pandas DataFrame."""
    data_dict = {}

    # Get number of nodes
    num_nodes = protein_data.num_nodes

    # Add basic node indices
    data_dict['node_idx'] = list(range(num_nodes))

    # Add all available node features
    for key in protein_data.keys:
        attr = getattr(protein_data, key)
        if attr is not None and hasattr(attr, 'shape') and attr.shape[0] == num_nodes:
            # Handle different feature shapes
            if len(attr.shape) == 1:  # Single value per node
                data_dict[key] = attr.tolist()
            elif len(attr.shape) == 2:  # Vector per node
                if key == 'amino_acid_one_hot':
                    # Convert one-hot to amino acid type
                    aa_types = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
                                'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
                    aa_indices = attr.argmax(dim=1).tolist()
                    data_dict['amino_acid'] = [aa_types[idx] if idx < len(aa_types) else 'UNK' for idx in aa_indices]
                else:
                    # For other vector features, we can take the mean or list them as strings
                    data_dict[key] = [str(attr[i].tolist()) for i in range(num_nodes)]

    # Create DataFrame
    df = pd.DataFrame(data_dict)
    return df

# Convert first protein to DataFrame and display
protein_df = protein_to_dataframe(train_ds[0])
print(protein_df.head(10))  # Show first 10 residues

In [ ]:
import os
import torch
import logging
from pathlib import Path
from tqdm import tqdm
from graphein.protein.graphs import construct_graph
from graphein.ml.conversion import GraphFormatConvertor
from graphein.ml.datasets.torch_geometric_dataset import ProteinGraphListDataset

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='protein_graph_processing.log'
)
logger = logging.getLogger(__name__)

# Get paths to all your PDB files
pdb_dir = os.path.expanduser("~/Downloads/nanobody_extracted2")
pdb_paths = [os.path.join(pdb_dir, f) for f in os.listdir(pdb_dir) if f.endswith('.pdb')]

# Create label map
label_map = {os.path.splitext(os.path.basename(path))[0]: 1 if "nanobody" in path else 0
             for path in pdb_paths}

# Initialize converter and lists for valid graphs
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg")
valid_graphs = []
valid_ids = []

# Process each file individually with error handling
print(f"Processing {len(pdb_paths)} PDB files...")
for path in tqdm(pdb_paths):
    struct_id = os.path.splitext(os.path.basename(path))[0]
    try:
        # Try to construct the graph with your config (use config_1A for maximum compatibility)
        g = construct_graph(config=config_1A, pdb_path=path)

        # Skip None graphs
        if g is None:
            logger.warning(f"Graph construction returned None for {struct_id}")
            continue

        # Convert to PyG format
        try:
            pyg_graph = convertor(g)

            # Add label if available
            if struct_id in label_map:
                pyg_graph.graph_y = torch.tensor([label_map[struct_id]], dtype=torch.long)

            # Store valid graph
            valid_graphs.append(pyg_graph)
            valid_ids.append(struct_id)
            logger.info(f"Successfully processed {struct_id}")

        except Exception as e:
            logger.error(f"Error converting graph for {struct_id}: {str(e)}")

    except Exception as e:
        logger.error(f"Error constructing graph for {struct_id}: {str(e)}")

# Create the dataset with only valid graphs
train_ds = ProteinGraphListDataset(
    root="data/",
    data_list=valid_graphs,
    name="train"
)

print(f"Dataset created with {len(train_ds)} protein graphs out of {len(pdb_paths)} PDB files")
print(f"Details logged to protein_graph_processing.log")

# Optional: Display summary stats about the dataset
if len(train_ds) > 0:
    first_graph = train_ds[0]
    print(f"Node feature dimensions: {first_graph.x.shape}")
    print(f"Available keys: {first_graph.keys}")

    # Count by label
    labels = torch.cat([data.graph_y for data in valid_graphs])
    label_counts = {l.item(): (labels == l).sum().item() for l in labels.unique()}
    print(f"Label distribution: {label_counts}")

In [ ]:
%rm -r data
